In [ ]:
# Note: you can skip this if you use the uv package manager in project root (/) directory
import subprocess
import sys
packages = [
    "crawl4ai>=0.2.0",
    "openai>=1.0.0",
    "pydantic>=2.0.0",
    "python-dotenv>=1.0.0",
    "requests>=2.25.0",
    "beautifulsoup4>=4.9.0"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

In [1]:

import pandas as pd
import numpy as np
import asyncio
import nest_asyncio
from typing import List, Dict, Any
import json
import re
import logging
import random
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig, CacheMode
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator

In [2]:
!uv run playwright install

You are using a frozen webkit browser which does not receive updates anymore on ubuntu20.04-x64. Please update to the latest version of your operating system to test up-to-date browsers.


In [3]:
class Scraper:
    def __init__(self, base_urls: List[str]):
        self.base_urls = base_urls
        self.crawler = AsyncWebCrawler()

    async def scrape_article(self, url: str) -> Dict[str, Any]:
        config = CrawlerRunConfig(
            js_code="window.scrollTo(0, document.body.scrollHeight);",
            wait_for="body",
            css_selector="article, main, body",
            markdown_generator=DefaultMarkdownGenerator(),
            cache_mode=CacheMode.WRITE_ONLY,
            exclude_external_links=True,
            exclude_social_media_links=True,
            page_timeout=25000,
            verbose=True
        )

        try:
            result = await self.crawler.arun(url=url, config=config)
            cleaned = self.clean_content(result.markdown.raw_markdown)
            title = result.metadata.get("title", url)
            return {"title": title, "url": url, "content": cleaned}
        except Exception as e:
            logging.error(f"Error scraping {url}: {e}")
            return {"title": url, "url": url, "content": "", "error": str(e)}

    async def scrape_multiple(self) -> List[Dict[str, Any]]:
        articles = []
        for url in self.base_urls:
            article = await self.scrape_article(url)
            articles.append(article)
            await asyncio.sleep(random.uniform(1, 2))
        return articles

    def clean_content(self, raw_markdown: str) -> str:
        if not raw_markdown:
            return ""
        content = raw_markdown
        content = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', content)
        content = re.sub(r'https?://\S+', '', content)
        content = re.sub(r'\[\d+\]', '', content)
        content = re.sub(r'[^a-zA-Z0-9\s.,;:!?\'\"()\-]', '', content)
        content = content.replace('\n', ' ')
        content = re.sub(r'\s+', ' ', content)
        return content.strip()



In [4]:
urls = [
    "https://www.forbes.com/advisor/business/employee-retention-strategies/",
    "https://www.oracle.com/asean/human-capital-management/employee-retention-strategies/",
    "https://www.onsurity.com/blog/best-employee-retention-strategies/",
    "https://hbr.org/2022/07/its-time-to-reimagine-employee-retention",
    "https://www.gallup.com/workplace/650174/employee-retention-depends-getting-recognition-right.aspx",
    "https://www.navigatewell.com/resources/blog-posts/blog-9-powerful-employee-retention-strategies-for-2025-and-beyond",
    "https://www.roberthalf.com/us/en/insights/management-tips/effective-employee-retention-strategies",
    "https://www.researchgate.net/publication/385266961_Best_Practices_for_Improve_Employee_Retention",
    "https://hbr.org/topic/subject/employee-retention",
    "https://www.quantumworkplace.com/future-of-work/employee-retention-case-study",
    ## Added extra from TAFEP case studies
    "https://www.tal.sg/tafep/resources/case-studies/2024/helping-careers-take-flight-for-greater-employee-engagement",
    "https://www.tal.sg/tafep/resources/case-studies/2024/how-an-sme-achieved-greater-employee-performance",
    "https://www.tal.sg/tafep/resources/case-studies/2024/flexibility-to-meet-diverse-needs-and-boost-retention",
    "https://www.tal.sg/tafep/resources/case-studies/2024/empowering-talent-flexible-work-solutions-for-optimal-performance",
    "https://www.tal.sg/tafep/resources/case-studies/2024/addressing-the-healthcare-labour-crunch-through-age-inclusive-practices",
    "https://www.tal.sg/tafep/resources/case-studies/2024/building-a-caring-people-first-culture-for-higher-retention-rates",
    "https://www.tal.sg/tafep/resources/case-studies/2024/welcoming-industry-4-0-through-age-inclusive-practices-to-achieve-business-growth",
    "https://www.tal.sg/tafep/resources/case-studies/2024/investing-in-diversity-can-yield-lower-attrition-rates",
    "https://www.tal.sg/tafep/resources/case-studies/2023/flexible-work-arrangements-are-key-to-hiring-and-retaining-talent",
    "https://www.tal.sg/tafep/resources/case-studies/2019/maybank---age-inclusive",
    "https://www.tal.sg/tafep/resources/case-studies/2019/aerospace-component-engineering-services-pte-ltd",
    "https://www.tal.sg/tafep/resources/case-studies/2020/city-developments-limited",
    "https://www.tal.sg/tafep/resources/case-studies/2020/rajah-tann-singapore-llp",
    "https://www.tal.sg/tafep/resources/case-studies/2020/yayasan-mendaki",
    "https://www.tal.sg/tafep/resources/case-studies/2020/republic-polytechnic",
    "https://www.tal.sg/tafep/resources/case-studies/2020/infineon-singapore",
    "https://www.tal.sg/tafep/resources/case-studies/2020/hsl-constructor",
    "https://www.tal.sg/tafep/resources/case-studies/2020/aviva---caring-more-achieving-more",
    "https://www.tal.sg/tafep/resources/case-studies/2019/ibm",
    "https://www.tal.sg/tafep/resources/case-studies/2019/rockwell-automation-singapore",
    "https://www.tal.sg/tafep/resources/case-studies/2019/rohei",
    "https://www.tal.sg/tafep/resources/case-studies/2019/ocbc",
    "https://www.tal.sg/tafep/resources/case-studies/2020/emergenetics-caelan-and-sage",
    "https://www.tal.sg/tafep/resources/case-studies/2019/maybank"
]

nest_asyncio.apply()

async def main():
    scraper = Scraper(base_urls=urls)
    await scraper.crawler.start()
    articles = await scraper.scrape_multiple()
    await scraper.crawler.close()
    return articles

articles = await main()
pd.DataFrame(articles).to_json("employee_retention_articles.json", orient="records", indent=2)
print("✅ Scraped and saved successfully!")

[INIT].... → Crawl4AI 0.7.6 

[FETCH]... ↓ https://www.forbes.com/advisor/business/employee-retention-strategies/                               |
✓ | ⏱: 1.79s 

[SCRAPE].. ◆ https://www.forbes.com/advisor/business/employee-retention-strategies/                               |
✓ | ⏱: 0.18s 

[COMPLETE] ● https://www.forbes.com/advisor/business/employee-retention-strategies/                               |
✓ | ⏱: 1.97s 

[COMPLETE] ● Database backup created at: /home/xiwen/.crawl4ai/crawl4ai.db.backup_20251102_224607 

...

[COMPLETE] ● https://www.tal.sg/tafep/resources/case-studies/2020/emergenetics-caelan-and-sage                    |
✓ | ⏱: 1.28s 

[FETCH]... ↓ https://www.tal.sg/tafep/resources/case-studies/2019/maybank                                         |
✓ | ⏱: 1.29s 

[SCRAPE].. ◆ https://www.tal.sg/tafep/resources/case-studies/2019/maybank                                         |
✓ | ⏱: 0.03s 

[COMPLETE] ● https://www.tal.sg/tafep/resources/case-studies/2019/maybank                                         |
✓ | ⏱: 1.33s 

✅ Scraped and saved successfully!
